In [2]:
%pip install scikit-learn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
data='https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df=pd.read_csv(data)

In [16]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [4]:
categorical=['lead_source',
 'industry',
 'employment_status',
 'location']
numerical=['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [5]:
for c in categorical:
    df[c]=df[c].fillna('NA')
for n in numerical:
    df[n]=df[n].fillna(0.0)

In [6]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [7]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
df_train.head()
y_full_train=df_full_train.converted.values
y_test=df_test.converted.values
y_train=df_train.converted.values
y_val=df_val.converted.values
del df_full_train['converted']
del df_test['converted']
del df_train['converted']
del df_val['converted']

In [17]:
for c in categorical:
    print(c)
    print(round(mutual_info_score(y_train, df_train[c]), 2))
    from sklearn.feature_extraction import DictVectorizer


lead_source
0.03
industry
0.0
employment_status
0.02
location
0.0


In [10]:
dv=DictVectorizer(sparse=False)
train_dict=df_train[categorical+numerical].to_dict(orient='records')
X_train=dv.fit_transform(train_dict)
val_dict=df_val[categorical+numerical].to_dict(orient='records')
X_val=dv.transform(val_dict)
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred=model.predict_proba(X_val)[:,1]
og_decision=(y_pred>=0.5)
(y_val==og_decision).mean()
#round((y_val==decision).mean(),2)

np.float64(0.645)

In [11]:
for c in df_train.columns:
    dv=DictVectorizer(sparse=False)
    train_dict=df_train[categorical+numerical].drop(c, axis=1).to_dict(orient='records')
    X_train=dv.fit_transform(train_dict)
    val_dict=df_val[categorical+numerical].drop(c, axis=1).to_dict(orient='records')
    X_val=dv.transform(val_dict)
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred=model.predict_proba(X_val)[:,1]
    decision=(y_pred>=0.5)
    print(c+":",(y_val==og_decision).mean()-(y_val==decision).mean())

lead_source: 0.0030000000000000027
industry: 0.0
employment_status: 0.0
location: 0.0
annual_income: -0.07899999999999996
number_of_courses_viewed: 0.0020000000000000018
interaction_count: 0.04400000000000004
lead_score: 0.0010000000000000009


In [14]:
dv=DictVectorizer(sparse=False)
train_dict=df_train[categorical+numerical].to_dict(orient='records')
X_train=dv.fit_transform(train_dict)
val_dict=df_val[categorical+numerical].to_dict(orient='records')
X_val=dv.transform(val_dict)
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred=model.predict_proba(X_val)[:,1]
    decision=(y_pred>=0.5)
    print(c,':',(y_val==decision).mean())

1e-06 : 0.598
1e-05 : 0.598
0.0001 : 0.613
0.001 : 0.645
